# ESMFold Microbatching 

Benchmarking **microbatching for ESMFold inference** on  protein sequences and comparing throughput/latency and peak GPU memory across microbatch sizes.

In [ ]:
import os, sys, importlib.util, torch

print("cwd:", os.getcwd())
print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

print("esm spec:", importlib.util.find_spec("esm"))
print("esm.esmfold spec:", importlib.util.find_spec("esm.esmfold"))

In [ ]:
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

pip_install([
    "fair-esm",
    "biopython",
    "requests",
    "tqdm",
    "pandas",
    "matplotlib",
])

print("Installs done.")

In [ ]:
import time, math, random, json
from dataclasses import dataclass
from typing import List, Tuple, Dict, Any, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

import torch

## 1) Download *real* protein sequences from UniProt 

In [ ]:
from pathlib import Path

def download_uniprot_fasta(
    out_path: str,
    organism_id: int = 9606,             
    length_min: int = 80,
    length_max: int = 600,
    size: int = 200,                     
    reviewed_only: bool = True,
) -> str:
    """
    Downloads real protein sequences from UniProtKB via REST.
    Saves FASTA to out_path and returns the path.
    """
    out_path = str(out_path)
    p = Path(out_path)
    p.parent.mkdir(parents=True, exist_ok=True)

    reviewed_q = "reviewed:true AND " if reviewed_only else ""
    query = f'{reviewed_q}(organism_id:{organism_id}) AND (length:[{length_min} TO {length_max}])'

  
    url = "https://rest.uniprot.org/uniprotkb/stream"
    params = {
        "format": "fasta",
        "query": query,
        "size": str(size),
    }

    print("Fetching from UniProt:", url)
    print("Query:", query)
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()

    fasta_text = r.text
    if not fasta_text.strip().startswith(">"):
        raise RuntimeError("Did not receive FASTA. Response head:\n" + fasta_text[:300])

    p.write_text(fasta_text)
    print(f"Saved FASTA to: {p}  ({len(fasta_text)/1024:.1f} KB)")
    return out_path

FASTA_PATH = "/content/uniprot_reviewed_human_len80_600.fasta"
download_uniprot_fasta(FASTA_PATH, organism_id=9606, length_min=80, length_max=600, size=200, reviewed_only=True)

In [ ]:
from Bio import SeqIO

VALID_AA = set(list("ACDEFGHIKLMNPQRSTVWY"))

def load_fasta_sequences(path: str) -> List[Tuple[str, str]]:
    """
    Returns list of (id, sequence) from FASTA.
    """
    seqs = []
    for rec in SeqIO.parse(path, "fasta"):
        sid = str(rec.id)
        s = str(rec.seq).upper().replace(" ", "").replace("\n", "")
        seqs.append((sid, s))
    return seqs

def clean_sequences(
    seqs: List[Tuple[str, str]],
    min_len: int = 80,
    max_len: int = 600,
    drop_nonstandard: bool = True,
) -> List[Tuple[str, str]]:
    cleaned = []
    for sid, s in seqs:
        if len(s) < min_len or len(s) > max_len:
            continue
        if drop_nonstandard and any(ch not in VALID_AA for ch in s):
            continue
        cleaned.append((sid, s))
    return cleaned

all_seqs = load_fasta_sequences(FASTA_PATH)
seqs = clean_sequences(all_seqs, min_len=80, max_len=600, drop_nonstandard=True)

print("Loaded:", len(all_seqs), "sequences")
print("Cleaned:", len(seqs), "sequences")
print("Example:", seqs[0][0], "len=", len(seqs[0][1]))

In [ ]:
random.seed(0)
N_BENCH = 64
if len(seqs) < N_BENCH:
    raise RuntimeError(f"Not enough cleaned sequences ({len(seqs)}) for N_BENCH={N_BENCH}")

bench = random.sample(seqs, N_BENCH)

lengths = [len(s) for _, s in bench]
print("Benchmark size:", len(bench))
print("Min/Median/Max length:", min(lengths), sorted(lengths)[len(lengths)//2], max(lengths))

## 2) Load ESMFold

In [ ]:
import esm

def load_esmfold():
    if hasattr(esm, "pretrained") and hasattr(esm.pretrained, "esmfold_v1"):
        model = esm.pretrained.esmfold_v1()
        return model


    try:
        from esm.esmfold.v1 import esmfold_v1
        model = esmfold_v1()
        return model
    except Exception as e:
        raise RuntimeError("Could not load ESMFold via known entry points.") from e

model = load_esmfold()
model = model.eval().cuda()
if hasattr(model, "set_chunk_size"):
    print("Model supports set_chunk_size(). Leaving it OFF for pure microbatching comparison.")
else:
    print("Model does NOT expose set_chunk_size()")

print("Model loaded on GPU.")

## 3) Bucketing + Microbatching

In [ ]:
def bucket_by_length(items: List[Tuple[str, str]], bucket_width: int = 50):
    """
    Group sequences into buckets by length range: [k, k+bucket_width).
    Returns list of buckets, each a list of (id, seq), sorted by length.
    """
    buckets: Dict[int, List[Tuple[str, str]]] = {}
    for sid, s in items:
        b = (len(s) // bucket_width) * bucket_width
        buckets.setdefault(b, []).append((sid, s))

    out = []
    for k in sorted(buckets.keys()):
        bucket = sorted(buckets[k], key=lambda x: len(x[1]))
        out.append(bucket)
    return out

def make_microbatches(bucket: List[Tuple[str, str]], microbatch_size: int):
    for i in range(0, len(bucket), microbatch_size):
        yield bucket[i:i+microbatch_size]

In [ ]:
@torch.no_grad()
def esmfold_infer_batch(seqs: List[str], use_bf16: bool = True):
    """
    Runs ESMFold inference on a list of sequences.
    We try common methods: infer(), infer_pdbs(), infer_pdb().
    Returns a lightweight object (e.g., lengths) to avoid huge memory retention.
    """
    autocast_ctx = torch.autocast(device_type="cuda", dtype=torch.bfloat16) if use_bf16 else torch.no_grad()

    with autocast_ctx:
        if hasattr(model, "infer"): 
            out = model.infer(seqs)
            return {"n": len(seqs), "lens": [len(s) for s in seqs]}

        if hasattr(model, "infer_pdbs"):  
            pdbs = model.infer_pdbs(seqs)
            return {"n": len(seqs), "lens": [len(s) for s in seqs]}

        if len(seqs) == 1 and hasattr(model, "infer_pdb"):
            pdb = model.infer_pdb(seqs[0])
            return {"n": 1, "lens": [len(seqs[0])]}

    raise RuntimeError("No supported ESMFold inference method found on this model object.")

## 4) Benchmark function

In [ ]:
@dataclass
class BenchResult:
    microbatch_size: int
    bucket_width: int
    use_bf16: bool
    n_seqs: int
    total_time_s: float
    seqs_per_s: float
    peak_mem_mib: float
    oom: bool = False
    error: str = ""

def benchmark_microbatching(
    items: List[Tuple[str, str]],
    microbatch_size: int,
    bucket_width: int = 50,
    use_bf16: bool = True,
    warmup_batches: int = 2,
) -> BenchResult:
    buckets = bucket_by_length(items, bucket_width=bucket_width)

    warm = []
    for b in buckets:
        warm.extend(b[:microbatch_size])
        if len(warm) >= microbatch_size * warmup_batches:
            break
    warm = warm[:microbatch_size]  
    if warm:
        _ = esmfold_infer_batch([s for _, s in warm], use_bf16=use_bf16)
        torch.cuda.synchronize()

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    t0 = time.perf_counter()
    n = 0

    try:
        for bucket in buckets:
            for mb in make_microbatches(bucket, microbatch_size):
                seq_list = [s for _, s in mb]
                _ = esmfold_infer_batch(seq_list, use_bf16=use_bf16)
                n += len(seq_list)

        torch.cuda.synchronize()
        t1 = time.perf_counter()

        peak = torch.cuda.max_memory_allocated() / (1024**2)
        total = t1 - t0
        sps = n / total if total > 0 else float("inf")

        return BenchResult(
            microbatch_size=microbatch_size,
            bucket_width=bucket_width,
            use_bf16=use_bf16,
            n_seqs=n,
            total_time_s=total,
            seqs_per_s=sps,
            peak_mem_mib=peak,
            oom=False,
        )

    except torch.cuda.OutOfMemoryError as e:
        torch.cuda.empty_cache()
        return BenchResult(
            microbatch_size=microbatch_size,
            bucket_width=bucket_width,
            use_bf16=use_bf16,
            n_seqs=n,
            total_time_s=float("nan"),
            seqs_per_s=float("nan"),
            peak_mem_mib=float("nan"),
            oom=True,
            error=str(e),
        )
    except Exception as e:
        return BenchResult(
            microbatch_size=microbatch_size,
            bucket_width=bucket_width,
            use_bf16=use_bf16,
            n_seqs=n,
            total_time_s=float("nan"),
            seqs_per_s=float("nan"),
            peak_mem_mib=float("nan"),
            oom=False,
            error=repr(e),
        )

In [ ]:
MICROBATCH_SIZES = [1, 2, 4, 8]
BUCKET_WIDTH = 50
USE_BF16 = True

results = []
for mb in MICROBATCH_SIZES:
    print(f"\n=== Microbatch size = {mb} ===")
    r = benchmark_microbatching(bench, microbatch_size=mb, bucket_width=BUCKET_WIDTH, use_bf16=USE_BF16)
    print(r)
    results.append(r)

df = pd.DataFrame([r.__dict__ for r in results])
df

In [ ]:
show = df.copy()
show["total_time_s"] = show["total_time_s"].round(3)
show["seqs_per_s"] = show["seqs_per_s"].round(3)
show["peak_mem_mib"] = show["peak_mem_mib"].round(1)
show[["microbatch_size", "n_seqs", "seqs_per_s", "total_time_s", "peak_mem_mib", "oom", "error"]]

## 5) Plot throughput and memory vs microbatch size

In [ ]:
import matplotlib.pyplot as plt

plot_df = show[~show["oom"]].copy()

plt.figure()
plt.plot(plot_df["microbatch_size"], plot_df["seqs_per_s"], marker="o")
plt.xlabel("Microbatch size")
plt.ylabel("Sequences / second")
plt.title("ESMFold Throughput vs Microbatch Size (Real UniProt Sequences)")
plt.grid(True)
plt.show()

plt.figure()
plt.plot(plot_df["microbatch_size"], plot_df["peak_mem_mib"], marker="o")
plt.xlabel("Microbatch size")
plt.ylabel("Peak GPU memory (MiB)")
plt.title("ESMFold Peak Memory vs Microbatch Size (Real UniProt Sequences)")
plt.grid(True)
plt.show()

In [ ]:
OUT_CSV = "/content/microbatching_results.csv"
df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
df

In [ ]:
sid0, seq0 = bench[0]
print("Using:", sid0, "len=", len(seq0))

@torch.no_grad()
def infer_one_pdb(sequence: str) -> str:
    # Try the available APIs
    if hasattr(model, "infer_pdb"):
        return model.infer_pdb(sequence)
    if hasattr(model, "infer_pdbs"):
        return model.infer_pdbs([sequence])[0]
    raise RuntimeError("This model build does not expose infer_pdb or infer_pdbs")

pdb = infer_one_pdb(seq0)
pdb_path = "/content/example_real_uniprot.pdb"
with open(pdb_path, "w") as f:
    f.write(pdb)

print("Wrote PDB:", pdb_path)
print(pdb[:400])